# Vector Search

Load the saved job chunks and embeddings, store them in a persistent
Chroma database, and test vector similarity retrieval.

Prerequisite: run 02_chunking_and_embeddings.ipynb to generate
job_chunks.jsonl and job_embeddings.npy.

Run cells in order with the project environment. CUDA is used for model inference.

## 1. Load saved chunks and embeddings

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import chromadb

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "data" / "processed"

chunks_df = pd.read_json(
    processed_dir / "job_chunks.jsonl",
    lines=True,
    convert_dates=False
)

embeddings = np.load(
    processed_dir / "job_embeddings.npy",
    allow_pickle=False
)

assert len(chunks_df) == embeddings.shape[0]
assert chunks_df["chunk_id"].is_unique
assert np.isfinite(embeddings).all()

print("Chunk records:", len(chunks_df))
print("Embedding shape:", embeddings.shape)

Chunk records: 3618
Embedding shape: (3618, 384)


## 2. Open the persistent Chroma collection

In [2]:
database_path = project_root / "data" / "indexes" / "chroma"

client = chromadb.PersistentClient(path=str(database_path))

collection = client.get_or_create_collection(
    name="job_chunks_e5_small_v1",
    embedding_function=None,
    configuration={"hnsw": {"space": "cosine"}},
    metadata={"embedding_model": "intfloat/multilingual-e5-small"}
)

print("Database path:", database_path.relative_to(project_root).as_posix())
print("Collection name:", collection.name)
print("Stored chunks:", collection.count())

Database path: data/indexes/chroma
Collection name: job_chunks_e5_small_v1


Stored chunks: 3618


## 3. Prepare IDs, documents, and metadata

In [3]:
metadata_columns = [
    "ID",
    "Job Title",
    "Company Name",
    "Job Category",
    "Job Level",
    "Job Location",
    "Publication Date",
    "Tags"
]

def prepare_metadata(row):
    metadata = {
        column: str(row[column])
        for column in metadata_columns
        if pd.notna(row[column])
    }

    metadata["chunk_index"] = int(row["chunk_index"])
    return metadata

chunk_ids = chunks_df["chunk_id"].tolist()
documents = chunks_df["chunk_text"].tolist()

metadatas = [
    prepare_metadata(row)
    for _, row in chunks_df.iterrows()
]

assert len(chunk_ids) == len(documents) == len(metadatas) == len(embeddings)

print("Prepared records:", len(chunk_ids))
print("First chunk ID:", chunk_ids[0])
print("First metadata record:", metadatas[0])

Prepared records:

 3618
First chunk ID: LF0001_chunk_000
First metadata record: {'ID': 'LF0001', 'Job Title': 'DIR, Equities Quant', 'Company Name': 'Merrill', 'Job Category': 'Data and Analytics', 'Job Level': 'Mid Level', 'Job Location': 'New York, NY', 'Publication Date': '2025-07-28T23:00:54Z', 'chunk_index': 0}


## 4. Insert records in batches

In [4]:
BATCH_SIZE = 500

for start in range(0, len(chunk_ids), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(chunk_ids))

    collection.upsert(
        ids=chunk_ids[start:end],
        embeddings=embeddings[start:end].tolist(),
        documents=documents[start:end],
        metadatas=metadatas[start:end]
    )

    print(f"Stored records through {end} of {len(chunk_ids)}", flush=True)

assert collection.count() == len(chunk_ids)

print("Total stored chunks:", collection.count())

Stored records through 500 of 3618

Stored records through 1000 of 3618


Stored records through 1500 of 3618


Stored records through 2000 of 3618


Stored records through 2500 of 3618


Stored records through 3000 of 3618


Stored records through 3500 of 3618


Stored records through 3618 of 3618


Total stored chunks: 3618


## 5. Verify a stored record

In [5]:
stored_record = collection.get(
    ids=[chunk_ids[0]],
    include=["documents", "metadatas", "embeddings"]
)

assert stored_record["ids"] == [chunk_ids[0]]
assert stored_record["documents"][0] == documents[0]
assert stored_record["metadatas"][0] == metadatas[0]

np.testing.assert_allclose(
    stored_record["embeddings"][0],
    embeddings[0],
    rtol=1e-5,
    atol=1e-6
)

print("Stored record verification passed.")
print("Verified chunk ID:", stored_record["ids"][0])

Stored record verification passed.
Verified chunk ID: LF0001_chunk_000


## 6. Load the query embedding model

In [6]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "intfloat/multilingual-e5-small",
    device="cuda"
)

print("Model device:", embedding_model.device)

W1028 13:54:17.680000 32612 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Could not reach the Hub (Cannot reach https://huggingface.co/api/models/intfloat/multilingual-e5-small: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'intfloat/multilingual-e5-small'.


Model device: cuda:0


## 7. Retrieve the closest chunks

In [7]:
query = "Python backend developer working with APIs and databases"

query_embedding = embedding_model.encode(
    ["query: " + query],
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False
)

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

result_rows = []

for rank, metadata in enumerate(results["metadatas"][0], start=1):
    result_rows.append({
        "rank": rank,
        "chunk_id": results["ids"][0][rank - 1],
        "job_id": metadata["ID"],
        "title": metadata["Job Title"],
        "company": metadata["Company Name"],
        "cosine_distance": results["distances"][0][rank - 1]
    })

display(pd.DataFrame(result_rows))

,rank,chunk_id,job_id,title,company,cosine_distance
0,1,LF0117_chunk_002,LF0117,"Backend Software Engineer, TikTok Data Ecosyst...",TikTok,0.139751
1,2,LF0247_chunk_001,LF0247,Backend Software Engineer - Global E-commerce ...,TikTok,0.141380
2,3,LF0117_chunk_001,LF0117,"Backend Software Engineer, TikTok Data Ecosyst...",TikTok,0.147130
3,4,LF0323_chunk_001,LF0323,Senior Software Engineer (Consumer - Growth & ...,Coinbase,0.148995
4,5,LF0341_chunk_001,LF0341,Research Senior Software Data/AWS Engineer,Waters,0.152887


## 8. Inspect retrieved evidence

In [8]:
for rank, document in enumerate(results["documents"][0], start=1):
    metadata = results["metadatas"][0][rank - 1]

    print(f"\n--- Result {rank}: {metadata['ID']} ---")
    print("Title:", metadata["Job Title"])
    print(document)


--- Result 1: LF0117 ---
Title: Backend Software Engineer, TikTok Data Ecosystem (Data Lake)
modeling;
- Proficiency in programming languages such as Java, C++, and Scala is essential, along with strong coding skills and the ability to troubleshoot effectively;
- Experience with other big data systems/frameworks like Hive, HBase, or Kudu is a plus;
- A willingness to tackle challenging problems without clear solutions, a strong enthusiasm for learning new technologies, and prior experience in managing large-scale data (in the petabyte range) are all advantageous qualities.
TikTok is committed to creating an inclusive space where employees are valued for their skills, experiences, and unique perspectives. Our platform connects people from across the globe and so does our workplace. At TikTok, our mission is to inspire creativity and bring joy. To achieve that goal, we are committed to celebrating our diverse voices and to creating an environment that reflects the many communities we re

## Vector-search observations

For the Python/backend/API/database query, five chunks represent four jobs because
LF0117 appears twice. LF0247 has the clearest combined backend, accepted-Python,
and database evidence. Other chunks are partial matches.

This example shows why results need source inspection and job deduplication.
It is not a complete quality evaluation.


## 9. Present distinct jobs

In [9]:
candidate_results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=20,
    include=["documents", "metadatas", "distances"]
)

job_rows = []
seen_job_ids = set()

for position, metadata in enumerate(candidate_results["metadatas"][0]):
    job_id = metadata["ID"]

    if job_id in seen_job_ids:
        continue

    seen_job_ids.add(job_id)

    job_rows.append({
        "rank": len(job_rows) + 1,
        "job_id": job_id,
        "title": metadata["Job Title"],
        "company": metadata["Company Name"],
        "best_chunk_id": candidate_results["ids"][0][position],
        "cosine_distance": candidate_results["distances"][0][position]
    })

    if len(job_rows) == 5:
        break

display(pd.DataFrame(job_rows))

,rank,job_id,title,company,best_chunk_id,cosine_distance
0,1,LF0117,"Backend Software Engineer, TikTok Data Ecosyst...",TikTok,LF0117_chunk_002,0.139751
1,2,LF0247,Backend Software Engineer - Global E-commerce ...,TikTok,LF0247_chunk_001,0.141380
2,3,LF0323,Senior Software Engineer (Consumer - Growth & ...,Coinbase,LF0323_chunk_001,0.148995
3,4,LF0341,Research Senior Software Data/AWS Engineer,Waters,LF0341_chunk_001,0.152887
4,5,LF0022,"Lead Cloud Automation Engineer (DevOps, AI/ML/...",EPAM Systems,LF0022_chunk_001,0.153270
